# 用 Sciverse 做科学 RAG 数据源

> 将 Sciverse 作为 RAG pipeline 的检索后端，为 LLM 提供可信科学证据

---

**Sciverse Cookbook** | [文档首页](https://sciverse.space/docs#cookbook) | [申请 API Key](https://sciverse.space/docs#auth)

## 前置准备

1. 在 [Sciverse 控制台](https://sciverse.space/) 申请 API Token
2. 安装依赖：`pip install httpx anthropic`
3. 将下方 `sv-...` 替换为你的真实 Token


In [ ]:
# 安装依赖（Colab 环境）
!pip install -q httpx anthropic


## Step 1: 定义 RAG 检索函数

封装 Sciverse agentic-search 为 RAG retriever


In [ ]:
import httpx

BASE = "https://api.sciverse.space"
TOKEN = "sv-..."
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

async def sciverse_retrieve(query: str, top_k: int = 10):
    """Sciverse RAG retriever"""
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            f"{BASE}/agentic-search",
            headers=HEADERS,
            json={"query": query, "top_k": top_k}
        )
        hits = resp.json()["hits"]
        # 格式化为 RAG context
        contexts = []
        for h in hits:
            contexts.append({
                "text": h["chunk"],
                "metadata": {
                    "doc_id": h["doc_id"],
                    "title": h["title"],
                    "score": h["score"]
                }
            })
        return contexts

contexts = await sciverse_retrieve("mRNA vaccine lipid nanoparticle delivery")
print(f"Retrieved {len(contexts)} evidence chunks")

## Step 2: 构建 RAG prompt

将检索结果注入 LLM prompt


In [ ]:
def build_rag_prompt(question: str, contexts: list) -> str:
    context_str = "\n\n".join([
        f"[{i+1}] {c['metadata']['title']}\n{c['text']}"
        for i, c in enumerate(contexts)
    ])
    return f"""基于以下科学文献证据回答问题。每个论点必须标注来源编号。
如果证据不足以回答，请明确说明。

## 证据
{context_str}

## 问题
{question}

## 回答（带引用）"""

prompt = build_rag_prompt(
    "LNP 递送 mRNA 的关键技术挑战有哪些？",
    contexts
)
print(prompt[:500] + "...")

## Step 3: 调用 LLM 生成回答

将 RAG prompt 传给 LLM 获取带引用的回答


In [ ]:
from anthropic import Anthropic

client = Anthropic()

msg = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=2048,
    messages=[{"role": "user", "content": prompt}]
)

answer = msg.content[0].text
print(answer)

---

## 下一步

- 📖 [查看完整 API 文档](https://sciverse.space/docs#sciverse/api)
- 🔑 [申请 / 管理 API Token](https://sciverse.space/)
- 📚 [浏览更多 Cookbook 案例](https://sciverse.space/docs#cookbook)
- 💬 [加入开发者社群](https://sciverse.space/)
